# **HuggingFace**

In [ ]:
# @title Install HuggingFace Libraries
#####################################

# ⚠️ Accelerator: Gemma-4-E4B in 4-bit quantization requires ~8–9 GB VRAM. Use e.g. A100 (40 GB) or L4 (24 GB)
# ⚠️ Gemma 4 support requires very recent transformers version. Don't just use 'pip install transformers'
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate bitsandbytes
!pip install -q -U peft trl datasets # Required for PEFT tuning
!pip install wandb -q -U
!pip install -U keras huggingface_hub -q

# Imports
# ⚠️ !!! Restart runtime (Runtime → Restart session) and verify version:
import transformers
import torch
from transformers import pipeline, BitsAndBytesConfig
import wandb
from transformers import pipeline, AutoProcessor, Gemma4ForConditionalGeneration, BitsAndBytesConfig, GenerationConfig
print(f'Transformers version: {transformers.__version__}')  # Should show something like 5.6.0.dev0

# Authentification (Colab Secrets or GCP Secrets to store HF access token)
from google.colab import userdata
from huggingface_hub import login
# Huggingface Token from Colab Secrets
# ⚠️ Create token on https://huggingface.co/settings/tokens and add 'Repositories permissions' for model 'google/gemma-4-E4B-it'
hf_token = userdata.get('HF_TOKEN')
# Log in to Hugging Face
login(token=hf_token)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.44.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have 

##### ✅ Inference

In [ ]:
# @title HuggingFace Inference (DeepSeek with FineGrainedFP8Config)
#####################################

model_id = "deepseek-ai/DeepSeek-V4-Flash"     # https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
print(f"✅ Loading model: {model_id}")

# Achtung: wir lassen BitsAndBytes komplett weg, da das Modell bereits von den Emtiwcklern im Format FP8 quantisiert ist (`FineGrainedFP8Config`). Die `transformers`-Bibliothek stellt fest, dass das Modell bereits als FP8-Version vorliegt, und es gibt keinen Grund es mit BitsAndBytes (zB `load_in_4bit=True`) zu komprimieren.
# Wichtiger Hinweis: Um FP8-Modelle effizient und ohne Fehler auszuführen, benötigst du in der Regel eine neuere Grafikkarte, die dieses Format auf Hardware-Ebene unterstützt (z. B. NVIDIA RTX 4000er-Serie, Ada Lovelace oder Hopper Architektur).
# Alternativ: Suche nach der reinen Basis-Version oder Instruct-Version des Modells, die **kein** `FP8`, `AWQ` oder `GPTQ` im Namen trägt.

pipe = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",
    model_kwargs={
        "dtype": torch.bfloat16,
    }
)

messages = [
    {"role": "system",  "content": "You are a helpful assistant."},
    {"role": "user",    "content": "What AI are you, and which advantages do you have over other AI models?"}
]

outputs = pipe(messages, max_new_tokens=500)
print(outputs[0]["generated_text"][-1]["content"])

✅ Loading model: deepseek-ai/DeepSeek-V4-Flash


config.json:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

[transformers] FP8 quantized models is only supported on GPUs with compute capability >= 8.9 (e.g 4090/H100), actual = `7.5`. We will default to dequantizing the model to bf16. Feel free to use a different quantization method like bitsandbytes or torchao


model.safetensors.index.json:   0%|          | 0.00/5.37M [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 46 files:   0%|          | 0/46 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# @title HuggingFace Inference (Gemma with BitsAndBytesConfig)
#####################################

#model_id = "mistralai/Mistral-7B-Instruct-v0.3" # https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
#model_id = "deepseek-ai/DeepSeek-V4-Flash"     # https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
#model_id = "google/gemma-4-E2B-it"
model_id = "google/gemma-4-E4B-it"
#model_id = "google/gemma-4-26B-A4B-it"
#model_id = "google/gemma-4-31B-it"

# https://www.marktechpost.com/2026/04/29/top-10-kv-cache-compression-techniques-for-llm-inference-reducing-memory-overhead-across-eviction-quantization-and-low-rank-methods/?amp

print(f"✅ Loading model: {model_id}")

# 2. Define Quantization Configuration to fit model on GPU
  # ⚠️ BitsAndBytesConfig via quantization_config is now mandatory for HuggingFace Transformers v5+!
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)
# Rule of thumb for memory requirements during quantization:
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)

# 3. Load and Quantize Model
  # Mistral is text-only → "text-generation". AutoTokenizer is used internally
  # (not AutoProcessor like in Gemma that handles new 'mm_token_type_ids' instead of only Tokens, since no multimodal support)
  # Define HuggingFace pipeline with updated 'dtype' and 'quantization_config'. It loads and wires:
    # Neural network weights of Gemma model (16 GB file)
    # Tokenizer that converts text into numbers the model understands
    # Processor handles multimodal inputs (text + images for Gemma 4)
pipe = pipeline(
    "text-generation",                 # ← text-only task for Mistral, "image-text-to-text" for Gemma 4
    model=model_id,                    # Download original model from HF (~30 GB in float32) compressed to local disc (16 GB in bfloat16) as unquantized file 'model.safetensors: 16.0G'
    device_map="auto",                 # GPU placement: weights loaded into VRAM, quantized on-the-fly to 4-bit NF4 format: 'Loading weights: 2130/2130'
    model_kwargs={
        "dtype": torch.bfloat16,
        "quantization_config": quant_config # Weights read from disk, then further quantized to 4-bit  NF4 format (~8–9 GB) during the GPU load.
    }
)

# Mistral uses plain strings for content (not list-of-dicts like Gemma 4)
messages = [
    {"role": "system",  "content": "You are a helpful assistant."}, # gemma: [{"type": "text", "text": "You are a helpful assistant."}]
    {"role": "user",    "content": "What AI are you, and which advantages do you have over other AI models?"}
]

outputs = pipe(messages, max_new_tokens=500)
print(outputs[0]["generated_text"][-1]["content"])

✅ Loading model: google/gemma-4-E4B-it


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces

I am a large language model, trained by Google.

As a large language model, my capabilities are constantly being refined and updated through ongoing training and development by Google's AI researchers.

**Instead of claiming absolute superiority, it's more accurate to describe the advantages and strengths inherent in the technology powering me:**

### Core Strengths and Advantages:

1. **Massive Scale and Breadth of Knowledge:** I have been trained on a colossal dataset encompassing a vast amount of text, code, and information from the public web. This allows me to have a very broad and deep understanding across countless subjects—from quantum physics and ancient history to modern programming frameworks and casual conversation.
2. **Contextual Coherence and Long-Form Generation:** I am designed to maintain context over very long interactions. This means I can follow complex, multi-turn conversations, understand nuance, and generate long, coherent, and stylistically consistent pieces of

In [ ]:
# @title HuggingFace Inference (Mistral)
#####################################

model_id = "mistralai/Mistral-7B-Instruct-v0.3" # https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
#model_id = "deepseek-ai/DeepSeek-V4-Flash"     # https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
#model_id = "google/gemma-4-E2B-it"
#model_id = "google/gemma-4-E4B-it"
#model_id = "google/gemma-4-26B-A4B-it"
#model_id = "google/gemma-4-31B-it"

# https://www.marktechpost.com/2026/04/29/top-10-kv-cache-compression-techniques-for-llm-inference-reducing-memory-overhead-across-eviction-quantization-and-low-rank-methods/?amp

print(f"✅ Loading model: {model_id}")

# 2. Define Quantization Configuration to fit model on GPU
  # ⚠️ BitsAndBytesConfig via quantization_config is now mandatory for HuggingFace Transformers v5+!
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)
# Rule of thumb for memory requirements during quantization:
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)

# 3. Load and Quantize Model
  # Mistral is text-only → "text-generation". AutoTokenizer is used internally
  # (not AutoProcessor like in Gemma that handles new 'mm_token_type_ids' instead of only Tokens, since no multimodal support)
  # Define HuggingFace pipeline with updated 'dtype' and 'quantization_config'. It loads and wires:
    # Neural network weights of Gemma model (16 GB file)
    # Tokenizer that converts text into numbers the model understands
    # Processor handles multimodal inputs (text + images for Gemma 4)
pipe = pipeline(
    "text-generation",                 # ← text-only task for Mistral, "image-text-to-text" for Gemma 4
    model=model_id,                    # Download original model from HF (~30 GB in float32) compressed to local disc (16 GB in bfloat16) as unquantized file 'model.safetensors: 16.0G'
    device_map="auto",                 # GPU placement: weights loaded into VRAM, quantized on-the-fly to 4-bit NF4 format: 'Loading weights: 2130/2130'
    model_kwargs={
        "dtype": torch.bfloat16,
        "quantization_config": quant_config # Weights read from disk, then further quantized to 4-bit  NF4 format (~8–9 GB) during the GPU load.
    }
)

# Mistral uses plain strings for content (not list-of-dicts like Gemma 4)
messages = [
    {"role": "system",  "content": "You are a helpful assistant."}, # gemma: [{"type": "text", "text": "You are a helpful assistant."}]
    {"role": "user",    "content": "What is Mistral AI and what are their main language models?"}
]

outputs = pipe(messages, max_new_tokens=500)
print(outputs[0]["generated_text"][-1]["content"])

✅ Loading model: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=

 Mistral AI is a cutting-edge AI company based in Paris, France, founded by Adrien Barrère, Frédéric Courrari, and Théophile Chenusau in 2020. The company is dedicated to developing large-scale AI models, with a primary focus on large language models (LLMs).

As of now, Mistral AI has developed two notable language models:

1. Mistral-1.5B: This is the first model developed by Mistral AI, with a parameter count of 1.5 billion. It was trained on a diverse range of data, including web pages, books, and other text sources, to generate human-like text.

2. Mistral-300B: This is a more recent model developed by Mistral AI, with a parameter count of 300 billion. It is one of the largest AI models ever created, and it was trained on a much larger dataset than the Mistral-1.5B model. The Mistral-300B model is intended to generate more detailed, nuanced, and contextually appropriate responses than its predecessor.

These models are used to develop various AI applications, such as chatbots, tran

##### ✅ Tuning

In [ ]:
# @title 1. Prepare Dataset
#####################################

from datasets import load_dataset
import re

"""
Good call to split it up. For the dataset, I'd actually not go with ultrachat_200k
for a first run — it's 200k multi-turn conversations, way overkill and slow on a single GPU with QLoRA. Better picks:

mlabonne/guanaco-llama2-1k — 1k samples, already chat-formatted, finishes in ~10 min on a single GPU. Perfect smoke test.
HuggingFaceH4/ultrachat_200k — only if you want a real run; subset it to a few thousand rows.

I'll go with mlabonne/guanaco-llama2-1k so your first end-to-end run actually completes.
Note it has a text field already wrapped in Llama-2 [INST] format, which happens to match Mistral's template —
convenient, but we'll re-format to messages so SFTTrainer applies Mistral's template properly.
"""

# Guanaco rows look like: "<s>[INST] question [/INST] answer </s><s>[INST] ... "
# We parse them back into role-tagged messages so SFTTrainer can apply Mistral's
# chat template cleanly. Mistral v0.3 supports only user/assistant (no system).
PATTERN = re.compile(r"\[INST\](.*?)\[/INST\](.*?)(?=<s>\[INST\]|</s>|$)", re.DOTALL)

def to_messages(example):
    turns = PATTERN.findall(example["text"])
    messages = []
    for user, assistant in turns:
        messages.append({"role": "user", "content": user.strip()})
        messages.append({"role": "assistant", "content": assistant.strip()})
    return {"messages": messages}

ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
ds = ds.map(to_messages, remove_columns=ds.column_names)
ds = ds.filter(lambda x: len(x["messages"]) >= 2)  # drop any parse failures

split = ds.train_test_split(test_size=0.05, seed=42)
dataset_train = split["train"]
dataset_test = split["test"]

# Save to disk so the training script can load without re-parsing
dataset_train.save_to_disk("./data/train")
dataset_test.save_to_disk("./data/test")

print(f"Train: {len(dataset_train)} | Test: {len(dataset_test)}")
print("Sample:", dataset_train[0]["messages"][:2])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/950 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Train: 950 | Test: 50
Sample: [{'content': 'Please give me the minimum code necessary to create a Phaser.js game within an HTML document. It should be a window of at least 1024x1024 pixels.', 'role': 'user'}, {'content': 'The minimum HTML code required to create a Phaser.js game within an HTML document with a window of at least 1024x1024 pixels would be:\n\n<!DOCTYPE html>\n<html>\n<head>\n    <script src="https://cdn.jsdelivr.net/npm/phaser@5.0.0/dist/phaser.js"></script>\n</head>\n<body>\n    <script>\n        const game = new Phaser.Game({\n            type: Phaser.AUTO,\n            width: 1024,\n            height: 1024,\n            scene: {\n                create: function() {\n                    // your game logic goes here\n                }\n            }\n        });\n    </script>\n</body>\n</html>\n\nThis code includes the Phaser.js library from a CDN and creates an instance of a Phaser.Game with the specified width and height. The create method of the scene is where you

In [ ]:
# @title 2. Load and Quantize Model
#####################################
from transformers import AutoTokenizer, AutoModelForCausalLM


"""
Heads up before the code: the linked Gemma tutorial uses a vision dataset (image + text), but Mistral-7B-Instruct-v0.3 is text-only.
You can't feed it images. You'll need either a text-only dataset (e.g. `HuggingFaceH4/ultrachat_200k`, `mlabonne/guanaco-llama2-1k`)
or to flatten the Gemma dataset to just its text fields. I'll write the code assuming `dataset_train`/`dataset_test` are text-only chat-formatted.

https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora

Main corrections to your Gemma-derived blocks:
- `Gemma4ForConditionalGeneration` → `AutoModelForCausalLM`
- `AutoProcessor` → `AutoTokenizer` (no images)
- Drop the vision-specific `collate_fn`, `skip_prepare_dataset`, and `dataset_text_field=""` — let SFTTrainer tokenize normally
- `processing_class=processor` → `processing_class=tokenizer`
- No `PatchedClippableLinear` monkey-patch needed — Mistral uses standard `nn.Linear`, so PEFT works out of the box

One more thing on the dataset: SFTTrainer auto-applies `tokenizer.apply_chat_template` if your dataset rows look like
`{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}`. Mistral v0.3's chat template uses
`[INST] ... [/INST]` markers and only supports `user`/`assistant` roles (no `system` — fold system content into the first user turn).
If your dataset has a `system` role, you'll get a template error.
"""

# Select model (https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Define Quantization Configuration to fit model on GPU
  # ⚠️ BitsAndBytesConfig via quantization_config is now mandatory for HuggingFace Transformers v5+!
  # BitsAndBytes shrink raw model from ~30 GB (in float32) to ~8–9 GB (in 4-bit precision) by quantization compression of weights
  # Transformers v5 updated weight-loading pipeline: 'load_in_4bit argument no longer top-level parameter to pass directly into
  # model's initialization. Instead, it must be wrapped in a BitsAndBytesConfig and passed via quantization_config.
  # Mistral 7B: ~14 GB on disk (bfloat16) → ~4 GB on GPU (4-bit quantized) - Much smaller than Gemma 4 E4B (16 GB → 8-9 GB)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Compresses weights from 16/32-bit → 4-bit (quantization) to fit 8.6 GB model on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,  # Precision for calculations: Upcast back to bfloat16 during math.
    bnb_4bit_quant_type="nf4",              # Quantization algorithm (NormalFloat4): Use NF4 format, best for normally-distributed LLM weights.
    bnb_4bit_use_double_quant=True          # Quantize quantization constants: adds second compression to save ~0.4 bits extra per parameter.
)
# Rule of thumb for memory requirements during quantization:
  # FP16 (standard): 2 bytes per parameter. (14 bytes × 2 = 28 GB)
  # 4-bit (quantized): approximately 0.7 to 0.8 bytes per parameter (including overhead). (14 bytes × 0.75 ≈ 10.5 GB)

# Load Tokenizer (text-only — no AutoProcessor since Mistral has no vision tower)
# Load Processor for Gemma (lightweight, handles text+image input preparation): 'processor = AutoProcessor.from_pretrained(model_id)'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Mistral has no pad token by default

# Load and Quantize Model (loads 14GB → quantizes to ~4GB on GPU). Gemma: 'Gemma4ForConditionalGeneration.from_pretrained'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=quant_config,
    device_map="auto")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Ja, das ist tatsächlich eine leicht verwirrende Namensgebung – "Transformers" ist sowohl die Architektur-Familie (das Paper "Attention is All You Need", 2017) als auch der Name der Hugging-Face-Bibliothek, die ursprünglich mal genau diese Architektur implementieren sollte. Mittlerweile deckt die Bibliothek aber weit mehr ab als nur Transformer-Architekturen (z. B. auch Mamba/State-Space-Modelle, CNNs für Vision), der Name ist historisch gewachsen und geblieben, obwohl er nicht mehr ganz präzise ist.

**Was die Bibliothek eigentlich ist**

**🤗 Transformers** ist im Kern eine einheitliche Python-API, um vortrainierte Modelle zu laden, zu nutzen, zu fine-tunen und zu teilen – egal ob NLP, Vision, Audio oder Multimodal. Die Kernidee: ein gemeinsames Interface für tausende verschiedene Modell-Architekturen und -Checkpoints.

**Die Kernkomponenten**

**Model-Klassen**
Jede Architektur (BERT, GPT, Llama, Qwen, etc.) hat eigene Klassen, aber mit konsistentem Interface. Für Inferenz/Fine-Tuning nutzt man meist die `AutoModel*`-Klassen, die automatisch die richtige Architektur anhand der Config laden:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
```

**Tokenizer**
Wie eben besprochen inklusive Chat-Template-Support (`apply_chat_template`). Es gibt "slow" (reines Python) und "fast" Tokenizer (Rust-basiert, aus der `tokenizers`-Bibliothek), letztere deutlich performanter.

**Pipelines**
Eine High-Level-Abstraktion für Standard-Tasks (Textgenerierung, Klassifikation, NER, Summarization), wenn man nicht selbst durch Tokenisierung → Forward-Pass → Decoding gehen will:

```python
from transformers import pipeline
generator = pipeline("text-generation", model="gpt2")
```

**Trainer-API**
Eine High-Level-Trainingsschleife (`Trainer`-Klasse) mit eingebauter Unterstützung für Mixed-Precision, Distributed Training, Gradient Accumulation, Logging etc. – man muss nicht den ganzen Trainings-Loop von Hand schreiben.

**Ökosystem drumherum**

Die Transformers-Bibliothek ist der Kern eines größeren Hugging-Face-Ökosystems, das für dich (gegeben deinen Research-Fokus auf Inference-Optimierung) relevant sein dürfte:

- **`accelerate`**: Abstraktion für Multi-GPU/TPU-Training und -Inferenz
- **`peft`**: Parameter-Efficient Fine-Tuning (LoRA, QLoRA, etc.)
- **`bitsandbytes`**: Integration für Quantisierung (u.a. LLM.int8(), das du ja bereits recherchiert hast)
- **`optimum`**: Hardware-spezifische Optimierungen (ONNX, TensorRT, etc.)
- **`tokenizers`**: die schnelle Rust-Implementierung
- **Hub**: die Plattform, auf der Modelle/Datasets/Spaces gehostet werden, gegen die die Bibliothek standardmäßig kommuniziert

**Ein Kritikpunkt, den man öfter hört**

Gerade im Kontext von Inferenz-Performance (dein Thema): Die Transformers-Bibliothek ist bewusst auf **Flexibilität und Forschungsfreundlichkeit** optimiert, nicht auf reine Inferenz-Geschwindigkeit. Deshalb greift man in Produktion für reine Inferenz oft zu spezialisierteren Frameworks wie **vLLM** (mit PagedAttention, kontinuierlichem Batching) oder **TGI** (Text Generation Inference, auch von Hugging Face, aber separat und inferenz-optimiert) statt direkt `model.generate()` aus Transformers in Produktion zu nutzen. Transformers selbst dient dabei aber oft als Referenzimplementierung, an der sich diese spezialisierten Engines orientieren bzw. deren Modell-Definitionen sie wiederverwenden.

`tokenizers` ist die Rust-basierte Bibliothek von Hugging Face, die für die eigentliche Tokenisierung zuständig ist – also das Zerlegen von Rohtext in die Tokens, die ein Modell tatsächlich verarbeitet. Sie ist von `transformers` unabhängig nutzbar, wird aber standardmäßig von den "fast" Tokenizern in `transformers` darunter verwendet.

**Warum eine separate Rust-Bibliothek?**

Die ursprünglichen Tokenizer in `transformers` waren reine Python-Implementierungen ("slow tokenizers"). Für große Datenmengen (z. B. beim Pretraining, wo man Milliarden Tokens verarbeiten muss) war das ein Bottleneck. `tokenizers` wurde daher in Rust geschrieben und über Bindings nach Python exponiert – typischerweise 10-100x schneller als die reinen Python-Implementierungen, mit paralleler Verarbeitung (Rayon für Multi-Threading) out of the box.

**Die Architektur: Tokenisierungs-Pipeline**

Das zentrale Konzept ist eine modulare Pipeline mit vier austauschbaren Komponenten:

**1. Normalizer**
Vorverarbeitung des Rohtexts – z. B. Unicode-Normalisierung (NFC/NFKC), Lowercasing, Whitespace-Bereinigung, Akzent-Entfernung.

**2. Pre-Tokenizer**
Grobe Vorsegmentierung, bevor das eigentliche Subword-Modell greift – meist Splitting an Whitespace/Punktuation, oder bei GPT-artigen Modellen ein Byte-Level-Splitting.

**3. Model (der eigentliche Tokenisierungs-Algorithmus)**
Hier steckt die eigentliche Subword-Tokenisierungs-Logik. Unterstützt werden u. a.:
- **BPE** (Byte-Pair Encoding) – iteratives Merging der häufigsten Zeichenpaare, genutzt von GPT-2, GPT-Neo, RoBERTa
- **WordPiece** – ähnlich BPE, aber mit likelihood-basiertem statt frequenzbasiertem Merge-Kriterium, genutzt von BERT
- **Unigram** – probabilistisches Modell, das vom Superset der möglichen Subwords iterativ die unwahrscheinlichsten entfernt, oft kombiniert mit SentencePiece
- **WordLevel** – simples Wort-zu-ID-Mapping (eher für einfache Fälle)

**4. Post-Processor**
Fügt Special Tokens hinzu (`[CLS]`, `[SEP]`, `<s>`, `</s>` etc.) und baut z. B. Segment-IDs für Sentence-Pairs auf.

**Byte-Level BPE (relevant für moderne LLMs)**

Die meisten aktuellen LLMs (GPT-Familie, Llama, Qwen, etc.) nutzen **Byte-Level BPE**: Statt auf Unicode-Zeichen-Ebene zu arbeiten, operiert der Tokenizer auf rohen Bytes (0-255). Vorteil: Es gibt garantiert kein Out-of-Vocabulary-Problem – jeder beliebige String lässt sich immer in eine Byte-Sequenz zerlegen, egal welche Sprache oder welches Symbol. Das ist besonders wichtig für multilinguale Modelle oder Code, wo man mit reinem Wort-Vocabulary schnell an Grenzen stößt.

**Training eines eigenen Tokenizers**

`tokenizers` erlaubt es, einen Tokenizer von Grund auf auf einem eigenen Korpus zu trainieren:

```python
from tokenizers import Tokenizer, trainers, models, pre_tokenizers

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=32000, special_tokens=["<s>", "</s>", "<pad>"])
tokenizer.train(files=["corpus.txt"], trainer=trainer)
```

Relevant z. B., wenn man ein Modell für eine Domäne mit stark abweichendem Vokabular trainiert (Code, biomedizinische Texte, andere Sprachen) – ein generisches Vocabulary verschwendet dort Tokens auf ineffiziente Zerlegungen.

**Relevanz für Inferenz-Performance**

Ein paar Punkte, die für deinen Fokus auf Inferenz-Optimierung interessant sein könnten:

- **Tokenisierungs-Effizienz ≠ Inferenz-Effizienz direkt**, aber die Vocabulary-Größe hat indirekten Einfluss: größeres Vocabulary → größere Embedding-Matrix und LM-Head, aber potenziell kürzere Sequenzen (weniger Tokens pro Text) → weniger Forward-Pass-Schritte. Es gibt hier einen Trade-off zwischen Speicherbedarf der Embedding-Schicht und Sequenzlänge.
- **DeepSeek-V3/V2** (im Kontext deiner MLA-Recherche) nutzt z. B. ein recht großes Vocabulary (128K Tokens), was die effektive Sequenzlänge für viele Sprachen reduziert – relevant, weil KV-Cache-Größe direkt von der Sequenzlänge abhängt, also indirekt mit der Tokenizer-Vocabulary-Größe zusammenhängt.
- **Offset-Mapping**: `tokenizers` gibt bei Bedarf zurück, welcher Text-Span zu welchem Token gehört (`return_offsets_mapping=True`) – nützlich für Tasks wie NER, wo man Token-Level-Predictions zurück auf den Originaltext mappen muss.

Falls interessant, könnte ich auch tiefer auf BPE- vs. Unigram-Tradeoffs eingehen, oder wie das genau mit den KV-Cache-Themen zusammenhängt, die du zuletzt recherchiert hast.

In [ ]:
# @title 3. Setup LoRA Configuration (train only ~1% of parameters)
#####################################

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare for quantized training.
model = prepare_model_for_kbit_training(model)

# Define LoRA config and wrap the full model.
lora_config = LoraConfig(
    r=16,                 # LoRA rank — higher = more capacity but more memory
    lora_alpha=32,        # Scaling factor (usually 2x rank)
    lora_dropout=0.05,    # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
    # Target modules to tune (attention layers + MLP layers for deeper memory changes)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",     # Attention layers
        "gate_proj", "up_proj", "down_proj"        # MLP layers (allows more capacity/ expressivity, but requires more VRAM)
        ],
    # modules_to_save=["lm_head", "embed_tokens"],  # Optional
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [ ]:
# @title 4. Analysis of Mistral Model
#####################################

# Analysis - Print Top level and inner submodules
print("\n ✅ Top-level submodules of the model:\n" + 40*"-")
for name, module in model.named_children():
    print(name, type(module).__name__)

print("\n ✅ Paths of inner submodules:\n" + 40*"-")
for name, module in model.model.named_children():
    print(name, type(module).__name__)

# Analysis - Inspect the actual LoRA matrix shapes
  # Forward pass is base(x) + (lora_B @ lora_A @ x) * (alpha/r). Seeing (16, 4096) and (4096, 16) makes it click why rank matters.
layer = model.base_model.model.model.layers[0].self_attn.q_proj
print("\n ✅ Inspect the actual LoRA matrix shapes:\n" + 40*"-")
print("base_layer:", layer.base_layer.weight.shape)   # (4096, 4096) frozen
print("lora_A:    ", layer.lora_A.default.weight.shape)  # (r, 4096)
print("lora_B:    ", layer.lora_B.default.weight.shape)  # (4096, r)

# Analysis - Quantify what LoRA actually costs
# For Mistral-7B with our config we should see ~0.4–0.5%. Then bump r from 16 → 64 and re-run — trainable params grow linearly with rank,
# and you can feel the memory/quality tradeoff.
"""
**Result:** 1.1% trainable is healthy and exactly what QLoRA is designed to deliver. Breaking down the 41.9M:

- Mistral-7B has 32 layers × 7 target modules (q/k/v/o + gate/up/down) = 224 LoRA-injected linear layers
- Each gets two matrices: `lora_A` of shape `(r, in_features)` and `lora_B` of shape `(out_features, r)`
- Roughly: `224 × r × (in + out)` ≈ 224 × 16 × ~11000 ≈ 39M, plus a bit of overhead → ~42M ✓

The "3.8B total" is interesting too — Mistral-7B has ~7.2B params, but `print_trainable_parameters` counts 4-bit packed weights
as roughly half (since two 4-bit values pack into one uint8), so you see ~3.8B. Total VRAM for those weights is ~4 GB, plus your
42M trainable params in bf16 (~84 MB), plus optimizer states for *only* those trainable params (AdamW = 2× params in fp32 ≈ ~340 MB),
plus activations. That's why QLoRA fits on a single GPU when full fine-tuning would need ~80 GB+. The takeaway: you're training 1%
of the params, using ~5% of the memory of a full fine-tune, and for instruction-following tasks you typically recover 90%+ of full-FT quality.
That's the whole pitch of QLoRA in one ratio.

!! --> Try `r=64` later and you'll see this jump to ~4.4% trainable — useful intuition for the rank/capacity tradeoff.
"""
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("\n ✅ Quantify what LoRA actually costs:\n" + 40*"-")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")


# Analysis - Confirm quantization actually happened
  # The base layer should be a Linear4bit, not a regular nn.Linear.
  # And check VRAM: torch.cuda.memory_allocated() / 1e9 right after model load — should be ~4–5 GB for Mistral-7B in 4-bit vs ~14 GB unquantized.
print("\n ✅ Confirm quantization actually happened:\n" + 40*"-")
print(type(layer.base_layer).__name__)   # Linear4bit
print(layer.base_layer.weight.dtype)     # torch.uint8 (packed 4-bit)

# Print all modules
"""
The contrast with your Gemma 4 dump is the whole point: Mistral has 32 layers (vs Gemma 4's 42), one flat layers ModuleList,
and crucially no vision_tower / audio_tower / embed_vision / embed_audio siblings. That's why target_modules=["q_proj", "k_proj", ...] works
without qualification — every q_proj in the model is a language-model attention projection,
so PEFT can't accidentally attach LoRA adapters to a vision encoder.

One small gotcha: if you run this after get_peft_model(model, lora_config), the structure changes — everything gets wrapped under
base_model.model.* and the Linear layers become lora.Linear. So run this snippet right after AutoModelForCausalLM.from_pretrained(...)
and before prepare_model_for_kbit_training / get_peft_model, otherwise the output will be much noisier and harder to read.
"""
# Mistral is text-only, so there's nothing to exclude for PEFT — but it's still
# useful to confirm the module paths so target_modules in LoraConfig are correct.
print("\n" + 60*"=" + "\n ✅ Print all named modules to see the exact structure:\n" + 60*"=")
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)

# Check which params are actually frozen
  # Sanity check that only LoRA weights have grads. You should see only lora_A, lora_B (and lora_magnitude_vector if DoRA were on, which it isn't here).
  # Everything else — base_layer, embed_tokens, lm_head, layernorms — should be frozen.
print("\n" + 60*"=" + "\n ✅ Check which params are frozen:\n" + 60*"=")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, tuple(p.shape))

# Look at the chat template itself
  # This is the part most people skip and then get confused by. You'll see Mistral's [INST] ... [/INST] markers and the </s> end-of-turn.
  # Understanding this explains why role formatting matters and why a template mismatch silently destroys fine-tuning quality.
print("\n" + 60*"=" + "\n ✅ Print chat template:\n" + 60*"=")
print(tokenizer.chat_template)
print(tokenizer.apply_chat_template(
    [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}],
    tokenize=False,
))


 ✅ Top-level submodules of the model:
----------------------------------------
base_model LoraModel

 ✅ Paths of inner submodules:
----------------------------------------
model MistralModel
lm_head Linear

 ✅ Inspect the actual LoRA matrix shapes:
----------------------------------------
base_layer: torch.Size([8388608, 1])
lora_A:     torch.Size([16, 4096])
lora_B:     torch.Size([4096, 16])

 ✅ Quantify what LoRA actually costs:
----------------------------------------
Trainable: 41,943,040 / 3,800,305,664 (1.1037%)

 ✅ Confirm quantization actually happened:
----------------------------------------
Linear4bit
torch.uint8

 ✅ Print all named modules to see the exact structure:
base_model.model.model.layers.0.self_attn.q_proj
base_model.model.model.layers.0.self_attn.q_proj.base_layer
base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.model.layers.0.self_attn.q_proj.lora_A
base_model.mo

In [ ]:
# @title 5. Run Tuning Job
#####################################

# Connect to wandb for Logging
"""
Watch the loss curve, not just the final number. In your SFTConfig, set report_to="wandb" and eval_strategy="steps",
eval_steps=20. A healthy QLoRA run on guanaco-1k should show train loss dropping from ~2.0 to ~1.2-ish over the epochs,
with eval loss tracking it. If eval loss starts climbing while train loss keeps falling — classic overfitting on a 1k dataset,
and a sign to lower epochs or r.

One thing to watch in the wandb dashboard specifically: the `train/grad_norm` chart. Healthy QLoRA runs sit between 0.5 and 2.0.
If you see spikes above 10, your LR is too high; if it's flatlined near zero, something's frozen that shouldn't be.
"""
from google.colab import userdata
import wandb, os

os.environ["WANDB_API_KEY"] = userdata.get("wandb")
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-qlora"

# Run Tuning Job
from trl import SFTConfig, SFTTrainer
from datasets import load_from_disk

# Load datasets BEFORE constructing the trainer
dataset_train = load_from_disk("./data/train")
dataset_test = load_from_disk("./data/test")

# Training Configuration
sft_config = SFTConfig(
    output_dir="./mistral-finetuned",
    num_train_epochs=4,                # ← 4 is too many for 950 samples. With ~950 train samples and effective batch 16, that's ~60 steps/epoch = ~180 total steps.
                                       #   4 epochs on a tiny dataset starts overfitting hard — eval loss U-turn around epoch 2-3.
    per_device_train_batch_size=4,     # ← A100 has 40/80 GB, bs=1 wastes it - we are using maybe 10 GB of 40/80 GB. With QLoRA + gradient checkpointing + seq_len 2048,
                                       #   bs=4 fits comfortably on 40 GB and bs=8 fits on 80 GB. If you OOM, drop back to 2.
    gradient_accumulation_steps=4,     # effective batch size = 1 × 4 = 4, or 4 x 4 = 16
    learning_rate=5e-5,                # ← better with 2e-4. Earlier set to '5e-5' was to reduce by 4x to stop oscillation in Gemma 4
    lr_scheduler_type="cosine",        # ← gradually decays LR as training progresses
    warmup_ratio=0.03,                 # short warmup avoids early-step LR shock (prevents early instability). Standard for QLoRA. Without warmup, first few steps with cosine can be unstable.
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",             # so wandb gets eval loss curve
    eval_steps=20,
    save_strategy="epoch",
    #save_total_limit=2,                  # ← don't fill disk with checkpoints
    #max_seq_length=2048,               # text-only — cap sequence length for memory --> I get an error message that this isn't supported!
    report_to="wandb",
    run_name="mistral-qlora-guanaco-r16",
    #gradient_checkpointing=True,         # ← trade ~20% speed for ~40% less VRAM (pushes batch size higher). Recomputes activations during backward instead of storing them.

)

# Train
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,         # optional but you have it
    processing_class=tokenizer,        # tokenizer for Mistral; would be `processor` for Gemma (vision)
    # peft_config=lora_config,         # not needed — model already wrapped via get_peft_model()
    # data_collator=collate_fn,        # not needed — text-only, SFTTrainer handles it
)
trainer.train()

# For ~950 training samples we will get 950/16 ≈ 60 steps/epoch × 4 epochs = ~240 steps total- Should finish under 30 min on an A100.
# Save LoRA Adapter (saves only small LoRA weights (~50 MB), not the full 4 GB model)
model.save_pretrained("./mistral-lora-adapter")
tokenizer.save_pretrained("./mistral-lora-adapter")  # for Gemma: 'processor.save_pretrained(...)'

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
20,1.243350,1.216247
40,1.143319,1.166273
60,1.135904,1.149961
80,1.078600,1.146883
100,1.040489,1.145283
120,0.940210,1.140463
140,0.875198,1.154915
160,0.920061,1.158224
180,1.033424,1.155719
200,0.906347,1.163924


('./mistral-lora-adapter/tokenizer_config.json',
 './mistral-lora-adapter/chat_template.jinja',
 './mistral-lora-adapter/tokenizer.json')

In [ ]:
# @title 6. Connect PEFT Layer to Base Model
#####################################

from peft import PeftModel

# 1. Load base model ('Gemma4ForConditionalGeneration.from_pretrained' for Gemma)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# Injecting PEFT Adapter - Load the PEFT adapter
model = PeftModel.from_pretrained(base_model, "./mistral-lora-adapter")
print("PEFT model loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PEFT model loaded successfully!


In [ ]:
# @title 7. Run Inference on Tuned Model
#####################################

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path = "./mistral-lora-adapter"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=quant_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# Mistral v0.3: no system role — fold any instruction into the first user turn
messages = [
    {"role": "user", "content": "What is Mistral AI and what are their main language models?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,                # ← unpack input_ids + attention_mask
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

# Slice off the prompt tokens so we only print the new response
response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,)
print(response)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Mistral AI is a cutting-edge artificial intelligence company based in Paris, France, with a mission to develop large language models that can understand and generate human-like text. Their main language models include:

- Laion: An open-source language model with 137 billion parameters. It is fine-tuned on a large dataset of internet text, including books, websites, and social media posts.

- Chat: A more advanced language model with 102 billion parameters. It is fine-tuned on a larger dataset of internet text and is designed to engage in human-like conversations.

- Muse: A language model with 500 billion parameters that is fine-tuned on a massive dataset of internet text, including books, websites, and social media posts. It is designed to generate high-quality, coherent, and engaging text.

These language models are used for a variety of applications, such as chatbots, language translation, and text generation.


ok, und die Verbindung zu Jinja, glaube ich ist hier, oder: 'inputs = tokenizer.apply_chat_template'

Genau, du hast den Nagel auf den Kopf getroffen. Die Verbindung läuft über den `PreTrainedTokenizer` (bzw. `PreTrainedTokenizerFast`) in `transformers`, der eigentlich zwei separate Dinge kombiniert:

**Die zwei getrennten Zuständigkeiten**

1. **`tokenizers`-Bibliothek**: zuständig für die eigentliche Zerlegung von Text in Token-IDs (BPE, WordPiece, etc.) — das haben wir eben besprochen.
2. **Jinja-Template**: zuständig für die Formatierung der *Konversationsstruktur* (Rollen, Sonderzeichen wie `<|user|>`) in einen einzigen rohen String, **bevor** überhaupt tokenisiert wird.

**Der konkrete Ablauf bei `apply_chat_template`**

```python
messages = [
    {"role": "system", "content": "Du bist hilfreich."},
    {"role": "user", "content": "Was ist die Hauptstadt von Frankreich?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,              # Standardwert
    add_generation_prompt=True,
    return_tensors="pt"
)
```

Intern passiert das nacheinander:

1. **Jinja-Rendering**: Das in der `tokenizer_config.json` gespeicherte `chat_template` (ein Jinja-String) wird mit den `messages` als Kontext gerendert → Ergebnis ist ein roher Text-String wie `<|system|>\nDu bist hilfreich.\n<|user|>\nWas ist die Hauptstadt...\n<|assistant|>\n`
2. **Tokenisierung**: Dieser String wird dann (falls `tokenize=True`, der Default) durch die eigentliche `tokenizers`-Pipeline geschickt → Token-IDs.

Wenn du nur den Zwischenschritt sehen willst, ohne zu tokenisieren:

```python
prompt_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt_string)
# Gibt den rohen, Jinja-gerenderten String aus
```

**Warum diese Trennung sinnvoll ist**

Jinja kümmert sich um die **Struktur/Formatierung** (welche Sonderzeichen, welche Reihenfolge, wie werden System-Prompts behandelt), während `tokenizers` sich um die **Zerlegung in Subwords/IDs** kümmert. Das sind orthogonale Probleme: Das Chat-Format ist modellspezifisch definiert vom Fine-Tuning (welche Marker wurden beim Instruction-Tuning verwendet), während die Subword-Tokenisierung von der Vocabulary/dem BPE-Merge-Set abhängt, das beim Pretraining gelernt wurde. Beides ist zwar an dasselbe Modell gekoppelt, aber konzeptionell unabhängig – daher macht es Sinn, sie als zwei getrennte Konfigurations-Assets (`chat_template` als Jinja-String, `vocab.json`/`merges.txt` bzw. `tokenizer.json` für die Tokenisierung) im selben Tokenizer-Verzeichnis zu speichern, statt sie zu vermischen.

Jinja ist eine Template-Engine für Python, die im LLM-Kontext vor allem für **Chat-Templates** verwendet wird – also für die Formatierung von Konversationen in das spezifische Text-Format, das ein Modell beim Training gesehen hat.

**Das Grundproblem**

Ein LLM sieht beim Training nicht "Rollen" wie System/User/Assistant als abstrakte Konzepte, sondern reinen Text mit bestimmten Sonderzeichen und Markern. Zum Beispiel könnte eine Konversation intern so aussehen:

```
<|system|>
Du bist ein hilfreicher Assistent.
<|user|>
Was ist die Hauptstadt von Frankreich?
<|assistant|>
Paris.
```

Jedes Modell (Llama, Mistral, Qwen, etc.) hat dabei sein eigenes Format mit eigenen Sonderzeichen. Damit man nicht für jedes Modell manuell Strings zusammenbauen muss, speichern Frameworks wie Hugging Face Transformers ein **Jinja-Template** direkt in der Tokenizer-Konfiguration (meist als `chat_template` in der `tokenizer_config.json`).

**Wie das aussieht**

Ein typisches Chat-Template in Jinja könnte so aussehen:

```jinja
{% for message in messages %}
    {% if message['role'] == 'system' %}
        <|system|>
{{ message['content'] }}
    {% elif message['role'] == 'user' %}
        <|user|>
{{ message['content'] }}
    {% elif message['role'] == 'assistant' %}
        <|assistant|>
{{ message['content'] }}
    {% endif %}
{% endfor %}
{% if add_generation_prompt %}
    <|assistant|>
{% endif %}
```

Man übergibt eine Liste von Nachrichten (als Python-Dicts mit `role` und `content`), und Jinja rendert daraus den korrekten Prompt-String für genau dieses Modell.

**Warum Jinja konkret?**

- **Logik im Template selbst**: Jinja unterstützt Schleifen, Bedingungen, Variablen – genug, um auch komplexere Fälle abzubilden (z. B. Tool-Calls, mehrere System-Nachrichten, verschachtelte Konversationen).
- **Portabilität**: Das Template wird als String zusammen mit dem Modell/Tokenizer ausgeliefert, sodass jede Inferenz-Bibliothek (transformers, vLLM, llama.cpp, Ollama etc.) dieselbe Formatierung reproduzieren kann, ohne modell-spezifischen Code zu schreiben.
- **Trennung von Logik und Code**: Die Chat-Format-Logik lebt als Konfigurationsdaten, nicht als hartcodierte Python-Funktion.

**Praktische Relevanz**

Wenn du z. B. mit Hugging Face arbeitest, nutzt du das oft über:

```python
tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

Das lädt intern das Jinja-Template aus der Tokenizer-Config und rendert deine Nachrichtenliste hinein. Ein häufiges Problem in der Praxis: Wenn das Chat-Template beim Fine-Tuning oder bei der Inferenz nicht exakt zum Trainings-Format passt (Whitespace, Sonderzeichen, Reihenfolge), kann die Modell-Performance spürbar leiden – das ist eine bekannte Fehlerquelle.

Gerade relevant für dich vermutlich auch im Kontext deiner Recherche zu Inference-Optimierung: Bei Themen wie MLA oder speziellen Attention-Varianten ändert Jinja nichts an der eigentlichen Architektur, aber ein falsch konfiguriertes Chat-Template ist oft die Ursache, wenn ein Modell in Benchmarks unerwartet schlecht performt, obwohl die Architektur eigentlich stimmt.

**Tokenizer Expansion** (auch "Vocabulary Extension") bezeichnet das Hinzufügen neuer Tokens zum Vocabulary eines bereits trainierten Tokenizers/Modells – meist um es an eine neue Sprache, Domäne oder Symbolik anzupassen, ohne komplett neu zu pretrainen.

**Das Grundproblem**

Ein Tokenizer wird beim Pretraining auf einem bestimmten Korpus trainiert (BPE-Merges lernen). Wenn das Modell danach auf Text trifft, für den das Vocabulary schlecht geeignet ist – z. B. eine Sprache, die im Trainingskorpus kaum vorkam, oder stark domänenspezifisches Vokabular (z. B. chemische Formeln, Code in einer seltenen Sprache) – zerfällt jedes Wort in sehr viele kleine Subword-Tokens. Das hat zwei Nachteile:

1. **Ineffizienz**: Mehr Tokens pro Wort → längere Sequenzen → höhere Rechenkosten, größerer KV-Cache
2. **Qualitätsverlust**: Extrem fragmentierte Subwords erschweren dem Modell, semantische Bedeutung zu erfassen

**Der klassische Anwendungsfall: Sprachanpassung**

Viele chinesische, japanische oder andere nicht-englische LLMs starten von einem englisch-dominierten Basismodell (z. B. Llama) und erweitern das Vocabulary gezielt um Tokens für die Zielsprache. Ein bekanntes Beispiel ist **Chinese-LLaMA**, das das ursprüngliche Llama-Vocabulary (das kaum chinesische Zeichen effizient abdeckte) um tausende chinesische Subwords/Zeichen erweitert hat.

**Wie es technisch funktioniert**

**1. Neues Vocabulary lernen**
Man trainiert entweder einen komplett neuen Tokenizer auf dem Zielsprachen-Korpus oder nutzt SentencePiece/BPE, um zusätzliche Merges/Tokens zu extrahieren, die im Originalvocabulary fehlen.

**2. Vocabulary mergen**
Die neuen Tokens werden zum bestehenden Vocabulary hinzugefügt (Vereinigungsmenge, meist ohne Duplikate).

**3. Embedding-Matrix erweitern**
Das ist der eigentlich kritische Schritt: Die Embedding-Matrix (Shape `[vocab_size, hidden_dim]`) und der LM-Head (oft mit den Input-Embeddings tied, also gleiche Gewichte) müssen um Zeilen für die neuen Tokens erweitert werden. Diese neuen Zeilen sind zunächst zufällig initialisiert oder – üblicher – über den Mittelwert der Embeddings der Subword-Bestandteile approximiert, aus denen das neue Token vorher bestanden hätte.

**4. Weitertrainieren (Continued Pretraining)**
Die neuen Embedding-Zeilen sind untrainiert und brauchen zusätzliches Training (oft mit eingefrorenem restlichem Modell zunächst, dann vollständigem Fine-Tuning), damit sie sinnvolle Repräsentationen lernen.

**Warum das für deine Recherche relevant ist**

Gegeben deinen Fokus auf chinesische Labs (DeepSeek, Kimi, Qwen) und Inferenz-Effizienz, zwei Verbindungspunkte:

- **Qwen** z. B. nutzt von Anfang an ein sehr großes, multilinguales Vocabulary (~150K Tokens), gerade um das Fragmentierungsproblem bei Chinesisch/Englisch/Code von vornherein zu vermeiden, statt es nachträglich per Expansion zu lösen. Das ist im Grunde die "Ground-up"-Alternative zur nachträglichen Tokenizer-Expansion.
- **Sequenzlängen-Effekt auf KV-Cache**: Ein Modell mit schlecht angepasstem Vocabulary braucht für denselben chinesischen Text deutlich mehr Tokens als eines mit gut abgestimmtem Vocabulary. Das heißt: bei gleicher "semantischer" Kontextlänge ist der KV-Cache-Speicherbedarf höher – ein Punkt, der direkt an deine MLA/GQA-Recherche anknüpft, weil eine größere Token-Anzahl den Cache-Druck erhöht, den genau diese Techniken zu reduzieren versuchen.

**Herausforderungen**

- **Tied Embeddings**: Wenn Input-Embeddings und LM-Head-Gewichte geteilt sind (üblich bei kleineren Modellen), muss man beim Expandieren beide synchron erweitern.
- **Katastrophales Vergessen**: Zu aggressives Weitertrainieren nach Expansion kann bereits gelernte Fähigkeiten in der Originalsprache verschlechtern, wenn man nicht sorgfältig mit Lernraten/Datenmischung umgeht.
- **Downstream-Kompatibilität**: Ein erweitertes Vocabulary bricht die Kompatibilität mit dem Original-Tokenizer – Checkpoints, die auf dem alten Vocabulary basieren, lassen sich nicht mehr direkt weiterverwenden.

Willst du tiefer rein, wie genau die Embedding-Initialisierung bei neuen Tokens funktioniert (Mean-Pooling vs. andere Methoden), oder eher wie sich Vocabulary-Größe konkret auf die KV-Cache-Rechnung auswirkt?

##### ✅ Tests

In [ ]:
# @title HuggingFace Inference (Qwen auf T4)
#####################################

model_id = "Qwen/Qwen2.5-7B-Instruct"     # https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
print(f"✅ Loading model: {model_id}")

# T4 = Turing (sm_75): keine BF16-Tensor-Core-Beschleunigung, daher float16 statt bfloat16.
# Falls du bei langen Sequenzen/Batches in Speicherprobleme läufst:
# model_kwargs={"dtype": torch.float16, "quantization_config": BitsAndBytesConfig(load_in_4bit=True)}

pipe = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",
    model_kwargs={
        "dtype": torch.float16,
    }
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What AI are you, and which advantages do you have over other AI models?"}
]

outputs = pipe(messages, max_new_tokens=500)
print(outputs[0]["generated_text"][-1]["content"])

✅ Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an advanced language model created by Alibaba Cloud, known as Qwen. While I can't directly compare myself to every other AI model, I can highlight some of the key features and advantages that set me apart:

1. **Multilingual Support**: I support multiple languages, including but not limited to Chinese, English, and many others, making me versatile for a global audience.

2. **Large Pre-Training Scale**: I was trained on a massive dataset, allowing me to understand complex nuances and generate more coherent and contextually appropriate responses.

3. **Fine-Tuning Capabilities**: I can be fine-tuned for specific tasks or domains, enhancing my performance in specialized areas like customer service, technical support, creative writing, etc.

4. **High-Quality Text Generation**: I am designed to produce high-quality, fluent, and natural-sounding text, which is particularly useful for applications requiring sophisticated language understanding and generation.

5. **Ethical Consideration

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Do we have AGI with AI models already?"}
]

outputs = pipe(messages, max_new_tokens=300)
print(outputs[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The term "AGI" stands for Artificial General Intelligence, which refers to a hypothetical type of artificial intelligence that has the ability to understand, learn, and apply its knowledge across a wide range of tasks at a level comparable to or exceeding that of a human. As of now, no AI model can be considered AGI.

Current AI systems, such as those used in natural language processing (e.g., GPT-3), computer vision (e.g., Vision Transformer), and other specialized tasks, are highly capable within their specific domains but lack the general intelligence and adaptability required for AGI. These systems excel at specific tasks but struggle with tasks outside their training scope or require significant retraining.

In summary, while we have made tremendous progress in AI, we do not yet have AGI with existing AI models.


**1) Code korrekt?**

Ja, absolut. `pipe(messages, max_new_tokens=300)` mit einer Liste von Chat-Dicts ist der Standardweg für Text-Generation-Pipelines mit Chat-Modellen – die Pipeline ruft intern `tokenizer.apply_chat_template()` auf. Kein Fehler hier.

Die Warnung ist harmlos: Qwens `generation_config.json` setzt standardmäßig `max_length=20`. Da du `max_new_tokens` explizit übergibst, gewinnt das (wie die Meldung selbst sagt) – reine Info, kein Handlungsbedarf.

**2) Warum ~2 Minuten für 300 Tokens?**

Das sind grob **2,5 Tokens/Sekunde** – für ein 7B-Modell rein auf GPU ist das sehr langsam. Wichtigste Frage dafür: **welches** Skript genau ist geladen?

Du schreibst "das vorherige Skript für reine Inference" – das war vermutlich mein allererstes Skript (Cell 1), das lädt das **volle fp16-Modell ohne Quantisierung**:

```python
model_kwargs={"dtype": torch.float16}
```

Das sind **~15,2 GB Gewichte auf einer 16-GB-T4** – extrem knapp. Kaum Puffer für KV-Cache, Aktivierungen und CUDA-Kontext-Overhead (üblich: 300–500 MB). Wenn `device_map="auto"` das nicht komplett auf die GPU packen kann, lagert `accelerate` automatisch einzelne Layer auf die CPU aus – und dann muss bei **jedem einzelnen Token** ein Teil der Gewichte über PCIe zwischen CPU und GPU kopiert werden. Das ist um Größenordnungen langsamer und würde die 2 Minuten locker erklären.

Diagnose direkt im Notebook:
```python
print(model.hf_device_map)
```
Steht da irgendwo `'cpu'` oder `'disk'` neben `'cuda:0'` → Bingo, das ist deine Antwort.

Unabhängig davon gelten auf der T4 noch strukturelle Bremsen:
- **Autoregressiv = sequenziell:** 300 Tokens = 300 einzelne Forward-Passes durch alle 28 Layer, nicht parallelisierbar (anders als das initiale Prompt-Processing)
- **Kein Flash Attention 2:** T4 ist Turing (sm_75), FA2 braucht Ampere+ (sm_80). Die Pipeline fällt auf eager/SDPA-Attention zurück – spürbar langsamer
- **Speicherbandbreite ist der Flaschenhals, nicht Rechenleistung:** bei Batch-Size 1 müssen pro Token alle ~7,6 Mrd. Parameter durch den Speicherbus. T4 (~320 GB/s) ist gegenüber A100 (~2 TB/s) über 6× langsamer – das dominiert bei Einzel-Sequenz-Decoding
- **Kein Batching** → GPU-Auslastung bleibt niedrig

**Praktischer Fix:** Für reine Inference die 4-bit-quantisierte Variante aus deinem QLoRA-Setup nutzen (wie Cell 2, nur ohne LoRA-Wrapping) – Footprint nur ~5,5 GB statt 15,2 GB, garantiert genug Platz für KV-Cache, kein Offloading mehr nötig. Klingt paradox (BitsAndBytes-4bit hat eigentlich Dequant-Overhead pro Matmul, spart also primär VRAM, nicht zwingend Zeit) – aber wenn dein aktuelles Problem CPU-Offloading ist, gewinnt die Quantisierung hier trotzdem klar, weil sie das Offloading von vornherein vermeidet.

In [ ]:
# Das zeigt dir für jeden Layer/Submodul, wohin er gemappt wurde:
# zB: {'model.embed_tokens': 0, 'model.layers.0': 0, ..., 'model.layers.15': 'cpu', ..., 'lm_head': 0}
# Steht irgendwo 'cpu' (oder 'disk') statt einer Zahl (0 = GPU 0) → das bestätigt das CPU-Offloading als Ursache der 2 Minuten.
print(pipe.model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 'cpu', 'model.layers.26': 'cpu', 'model.layers.27': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


In [ ]:
# Zusätzlich, falls du es noch direkter sehen willst, wie viel VRAM tatsächlich belegt ist:
# Bei ~15 GB allocated auf einer 16-GB-T4 (abzüglich CUDA-Kontext-Overhead)
# ist es sehr wahrscheinlich, dass zumindest ein Teil ausgelagert wurde.
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Allocated: 12.75 GB
Reserved:  13.87 GB


Bestätigt sich vollständig: **3 von 28 Layern + `norm` + `rotary_emb` + `lm_head` liegen auf CPU.** Das erklärt die 2 Minuten.

**Warum das so brutal langsam ist:**

Bei autoregressiver Generierung läuft der Forward-Pass sequenziell durch alle Layer, Layer für Layer, bei **jedem einzelnen** der 300 generierten Tokens. Für jeden Token, der durch Layer 25–27 muss:
1. Aktivierung von GPU → CPU kopieren (PCIe)
2. Auf CPU rechnen (keine Tensor Cores, viel langsamere Matmuls)
3. Ergebnis zurück CPU → GPU kopieren (PCIe)

Das passiert 300× hintereinander, plus PCIe-Latenz kommt on top zur reinen Rechenzeit.

Besonders schmerzhaft ist **`lm_head` auf CPU**: Das ist die finale Projektionsmatrix `hidden_size × vocab_size` = `3584 × ~152.000` ≈ 545 Mio. Parameter – bei Qwens großem Vokabular eine der größten einzelnen Matrizen im ganzen Modell. Diese Matmul läuft bei **jedem** Token, weil sie die Logits fürs nächste Token berechnet – und sie läuft komplett auf CPU statt GPU.

Passt auch zu deinen Speicherzahlen: `12.75 GB allocated` auf der GPU + die ausgelagerten Layer/lm_head on top ≈ die ~15 GB fp16-Gewichte, die nicht mehr komplett in die 16 GB der T4 passen (abzüglich CUDA-Kontext-Overhead).



**Fix: 4-bit-Quantisierung für Inference** (analog zu deinem QLoRA-Setup, nur ohne LoRA-Wrapping) im folgenden.  Bei ~5,5 GB statt ~15 GB Footprint passt jetzt das komplette Modell (inkl. `lm_head`) plus KV-Cache-Puffer locker auf die T4 – kein Offloading mehr nötig. Erwartbar sind damit eher **Sekunden statt Minuten** für 300 Tokens.

In [ ]:
# @title HuggingFace Inference (Qwen auf T4, 4-bit quantisiert)
#####################################
from transformers import BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"✅ Loading model: {model_id}")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 = Turing, kein natives BF16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

pipe = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",
    model_kwargs={
        "quantization_config": quant_config,
    }
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Do we have AGI with AI models already?"}
]

outputs = pipe(messages, max_new_tokens=300)
print(outputs[0]["generated_text"][-1]["content"])

# Diagnose zur Kontrolle:
print(pipe.model.hf_device_map)   # sollte jetzt überall '0' zeigen, kein 'cpu' mehr

Dank dir! Es ist sogar gut, dass wir das model sofort mit ‚quant_config‘ laden, weil ich es dann gleich weiter für tuning nutzen kann, nach einem initialen inference test, oder?

Allerdings frage ich mich, warum ich (vor einem halben jahr für mistral) überhaupt den reinen inference schritt ohne quantisierung gemacht hatte. Gibt es da irgendeinen harten fachlichen grund, das nicht zu machen, wenn ich nur inference machen will?

**1) Ja, genau – aber ein technisches Detail dazu:**

`pipe.model` ist ein ganz normales `PreTrainedModel`-Objekt, du kannst es direkt weiterverwenden:

```python
model = pipe.model
tokenizer = pipe.tokenizer   # brauchst du separat für SFTTrainer/Dataset-Vorbereitung

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
```

Kein Neuladen nötig, der Inference-Test vorher "verbraucht" oder verändert das quantisierte Basismodell nicht. In der Praxis ist es aber oft sauberer, für den Tuning-Teil das Modell **direkt** über `AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config, ...)` zu laden statt über `pipeline(...)` – die Pipeline ist primär ein Inference-Convenience-Wrapper (Tokenisieren, Generieren, Decodieren in einem Aufruf), für Training brauchst du dieses Drumherum nicht, nur `model` und `tokenizer`. Beides funktioniert, aber wenn du ohnehin weißt, dass Tuning der nächste Schritt ist, ist der direkte Load oft der geradlinigere Startpunkt.

**2) Kein harter fachlicher Grund – aber ein paar echte Gründe, die dafür sprechen können:**

Es gibt keine Regel "Inference darf nicht quantisiert laufen". Legitime Gründe, bei purer Inference trotzdem unquantisiert zu bleiben:

- **Qualität/Präzision:** NF4-Quantisierung verliert etwas Genauigkeit gegenüber fp16 – bei Chat-Antworten meist kaum spürbar, bei präzisionslastigen Aufgaben (Mathe, Code, exaktes Reasoning) oder wenn du Benchmark-Zahlen für das *Basismodell* reproduzierbar messen willst, kann das relevant werden.
- **Geschwindigkeit bei ausreichend VRAM:** Kontraintuitiv, aber BitsAndBytes-4bit ist nicht automatisch schneller als fp16 – bei jeder Matmul muss erst von 4-bit zurück auf compute_dtype dequantisiert werden. Auf Turing (T4) gibt's dafür keine hardwarebeschleunigten Kernels wie auf Ampere+, das kostet Overhead. Wenn genug VRAM da ist, kann fp16 pro Token sogar schneller sein.
- **Batching/Throughput:** Mit mehr VRAM-Puffer (weil unquantisiert eh schon viel Platz braucht, aber bei größeren GPUs kein Problem) lässt sich Batch-Size/Kontextlänge hochfahren, ohne den Dequant-Schritt pro Matmul mit reinzuziehen.

**Der konkrete Grund, warum es bei dir vor einem halben Jahr für Mistral einfach "funktioniert hat":** Das war wahrscheinlich kein bewusster fachlicher Entscheid, sondern reine VRAM-Arithmetik. Mistral-7B in fp16 liegt bei ~14 GB – auf einer 16-GB-T4 passt das mit ein bisschen Puffer für KV-Cache. Kein Offloading nötig, also keine Notwendigkeit, überhaupt über Quantisierung nachzudenken.

Bei Qwen2.5-7B ist der entscheidende Unterschied das **~5x größere Vokabular** (~152k vs. Mistrals ~32k Tokens). Das macht `embed_tokens` und – da bei den größeren Qwen2.5-Modellen `lm_head` **nicht** mit den Embeddings geteilt wird (untied) – auch `lm_head` deutlich größer. Grob geschätzt kommen dadurch mehrere hundert Millionen Extra-Parameter allein aus den Vokabular-Layern zusammen, was den fp16-Footprint auf ~15,2 GB drückt statt Mistrals ~14 GB. Genau dieses letzte Gigabyte war knapp genug, um dich über die 16-GB-Grenze zu schieben → Offloading → die 2 Minuten.

Kurz: Es war nicht "Mistral unquantisiert ist fachlich richtig, Qwen quantisiert ist ein Kompromiss" – es war einfach Zufall der Zahlen. Bei genug VRAM (z. B. A100/L4 mit 24+ GB) hättest du auch Qwen problemlos unquantisiert laufen lassen können, ganz ohne Qualitätskompromiss.

Mini-Guide: LLM Inference & Tuning mit 🤗 Transformers auf T4

Praxisleitfaden für den Umgang mit 7B-Modellen (Mistral, Qwen2.5, ...) auf einer 16-GB-T4-GPU.

---

1. Grundsatzentscheidung: Inference vs. Tuning

| | **Inference** | **Tuning (LoRA/QLoRA)** |
|---|---|---|
| **API** | `pipeline("text-generation", ...)` | `AutoModelForCausalLM.from_pretrained(...)` |
| **Warum** | Übernimmt Tokenisieren, Generieren, Decodieren in einem Aufruf – genau das braucht man für Inference | `pipeline()` bringt unnötigen Convenience-Overhead; Training braucht nur `model` + `tokenizer` direkt |
| **Quantisierung** | Nur falls nötig (siehe Abschnitt 2) | Praktisch immer (QLoRA-Trick, um Training auf begrenztem VRAM möglich zu machen) |

> **Wichtig:** Quantisierung ist kein inhärentes Erfordernis von Tuning an sich – auf einer A100/H100 mit viel VRAM macht man Full-Fine-Tuning oder LoRA oft auch unquantisiert. QLoRA ist der Kompromiss für limitierte Hardware wie die T4.

**Von Inference zu Tuning wechseln, ohne neu zu laden:**
```python
model = pipe.model
tokenizer = pipe.tokenizer   # separat holen, pipe selbst reicht dafür nicht

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
```
Funktioniert nur, wenn `pipe` bereits mit `quantization_config` geladen wurde.

---

2. Quantisierung: Ja oder Nein?

Der Auslöser ist **nicht** ein einzelner Faktor wie Vokabulargröße, sondern schlicht: **passt das Modell in fp16 komplett auf die GPU?**

```python
print(pipe.model.hf_device_map)   # oder: print(model.hf_device_map)
```
Taucht irgendwo `'cpu'` oder `'disk'` statt einer GPU-Nummer auf → `accelerate` lagert Layer aus. Das kostet bei **jedem einzelnen Token** einen PCIe-Roundtrip GPU↔CPU – kann Faktor 10-50x langsamer sein.

**Faustregel:**
- Modell passt komplett in fp16 auf die GPU → **kein** `quantization_config` nötig, ist oft sogar schneller (kein Dequant-Overhead pro Matmul)
- Modell ist zu groß (Offloading droht/passiert) → `BitsAndBytesConfig(load_in_4bit=True, ...)`

**Warum das modellabhängig variiert:** Nicht nur Parameterzahl zählt, sondern die Summe aus Layern, hidden_size *und* Vokabulargröße. Ein größeres Vokabular (z. B. Qwen2.5: ~152k vs. Mistral: ~32k Tokens) vergrößert `embed_tokens` und – falls **untied** – auch `lm_head` spürbar. Das kann ein Modell knapp über die VRAM-Grenze schieben, obwohl die "Kern"-Parameterzahl ähnlich ist.

**Wenn BitsAndBytes-NF4 trotz genug VRAM zu langsam läuft:** Vorquantisierte Checkpoints wie `-AWQ` oder `-GPTQ` nutzen optimierte Kernels ohne Live-Dequantisierung – auf älteren GPUs (Turing) oft spürbar schneller, dafür weniger flexibel als `load_in_4bit=True`.

---

3. T4-spezifische Stolperfallen (Turing, sm_75)

| Thema | Auf T4 richtig | Warum |
|---|---|---|
| **Precision** | `torch.float16` | Kein natives BF16 über Tensor Cores (erst ab Ampere/sm_80). `bfloat16` läuft, aber ohne Beschleunigung |
| **Attention** | `sdpa` (Default-Fallback) | Flash Attention 2 braucht Ampere+; auf T4 nicht verfügbar |
| **Speicherbandbreite** | Haupt-Flaschenhals bei Batch-Size 1 | ~320 GB/s vs. ~2 TB/s bei A100 – bei jedem Token müssen alle Parameter durch den Speicherbus |

---

4. Tokenizer vs. Processor

| Modelltyp | Klasse | Beispiele |
|---|---|---|
| Text-only | `AutoTokenizer` | Mistral, Qwen2.5 (Text-Varianten) |
| Vision/Multimodal | `AutoProcessor` | Gemma (Vision-Varianten), Qwen-VL |

**Trigger für die Entscheidung:** Hat das Modell einen `vision_tower`/`audio_tower`? Nicht das Modell-Alter ist relevant, sondern ob es multimodal ist.

---

5. `padding_side` – klassischer Stolperstein

| Kontext | Setting | Warum |
|---|---|---|
| Training (SFTTrainer, causal LM) | `"right"` (meist Default) | Standard für Trainingsläufe |
| Batch-Inference via `model.generate()` | `"left"` | Sonst sagt das Modell das nächste Token nach den Pad-Tokens vorher statt nach der echten Sequenz → Unsinn |

`pipeline()` kümmert sich bei Einzelanfragen meist selbst darum – sobald man für Batch-Inference manuell auf `model.generate()` umsteigt, muss man das selbst setzen.

---

6. Kurz-Checkliste vor dem Laden

- [ ] Text-only oder multimodal? → `AutoTokenizer` oder `AutoProcessor`
- [ ] Passt das Modell in fp16 auf die verfügbare GPU? → `hf_device_map` nach dem Laden prüfen
- [ ] Falls Offloading droht → `quantization_config` (NF4) oder vorquantisierten Checkpoint (AWQ/GPTQ) verwenden
- [ ] GPU-Architektur prüfen → fp16 statt bf16 auf Turing (T4); bf16 ok ab Ampere+
- [ ] Ziel Inference oder Tuning? → `pipeline()` vs. `AutoModelForCausalLM.from_pretrained()`
- [ ] Bei Batch-Inference mit `generate()`: `padding_side="left"` nicht vergessen

Oh cool danke und das war mir nicht klar: Full-Fine-Tuning oder LoRA oft auch unquantisiert. QLoRA ist der Kompromiss für limitierte Hardware wie die T4.

Ich habe auch einen macbook mit dem M5 chip und 48 GB RAM + 1TB speicher. Muss ich da noch quantisieren? Oder hängt das dann von der modelgrösse ab?

---

Guter Punkt, und die Antwort ist: **es hängt von der Modellgröße ab, aber du hast strukturell viel mehr Puffer als auf der T4** – aus einem architektonischen Grund, der die ganze Rechnung ändert.

**Der entscheidende Unterschied: Unified Memory**

Bei der T4 gibt's eine harte Trennung: 16 GB VRAM auf der GPU, separater Host-RAM daneben. Reicht das VRAM nicht, muss `accelerate` Layer über PCIe auf die CPU auslagern – das war genau dein Bottleneck bei Qwen.

Bei Apple Silicon (M5) teilen sich CPU und GPU denselben physischen Speicher (Unified Memory Architecture). Es gibt keine "GPU hat X GB, Rest ist getrennt" – deine 48 GB sind (abzüglich macOS-Overhead und laufender Apps) potenziell komplett für ein Modell nutzbar, ohne den PCIe-Copy-Roundtrip, der bei dir die 2 Minuten verursacht hat.

**Grobe Faustregel für fp16, bezogen auf deine 48 GB:**

| Modellgröße | fp16-Footprint | Passt unquantisiert? |
|---|---|---|
| 7B (Mistral, Qwen2.5-7B) | ~14 GB | Ja, viel Luft für KV-Cache |
| 13B | ~26 GB | Ja, sollte noch gut passen |
| 30-34B | ~60-68 GB | Nein – zu groß für 48 GB |
| 70B | ~140 GB | Nein, auch nicht annähernd |

macOS reserviert selbst etwas RAM und begrenzt normalerweise, wie viel die GPU als "working set" bekommt (nicht die vollen 48 GB) – praktisch nutzbar sind eher ~36-40 GB, je nach macOS-Version und laufenden Apps. Für 7B/13B-Modelle ist das aber komplett irrelevant, da ist massig Puffer.

**Praktisch wichtiger Haken:** Der ganze `BitsAndBytesConfig`-Workflow aus unseren Skripten läuft **nicht** auf dem Mac – BitsAndBytes ist CUDA-only, kein Metal/MPS-Support. Falls du bei größeren Modellen (30B+) doch quantisieren musst oder willst, brauchst du auf dem Mac einen anderen Stack:

- **MLX** (Apples eigenes Framework, `mlx-lm`) – nativ für Apple Silicon optimiert, hat 4-bit/8-bit-Quantisierung eingebaut und unterstützt auch LoRA/QLoRA-Fine-Tuning direkt. Aktuell wahrscheinlich der geradlinigste Weg für dich, sowohl Inference als auch Tuning lokal.
- **llama.cpp / GGUF** – sehr ausgereift für quantisierte Inference über Metal, viele Modelle als GGUF verfügbar, aber eher Inference-fokussiert, nicht Training.
- **PyTorch mit MPS-Backend** (`device="mps"`) – funktioniert für viele HF-Modelle, aber du müsstest `device_map="auto"` durch explizites `.to("mps")` ersetzen, und es gibt bei manchen Operationen/Datentypen noch Lücken in der MPS-Op-Coverage gegenüber CUDA.

Kurz: Für 7B/13B in fp16 → keine Quantisierung nötig, läuft direkt. Sobald du in Richtung 30B+ willst, brauchst du Quantisierung, aber über MLX oder GGUF statt BitsAndBytes.

###### analysis

Gute Nachricht vorweg: Qwen2.5-7B-Instruct folgt architektonisch dem gleichen Llama-Stil wie Mistral (q_proj/k_proj/v_proj/o_proj + gate_proj/up_proj/down_proj), also lässt sich fast alles 1:1 übernehmen. Ein paar Stellen müssen aber angepasst werden – vor allem wegen des größeren Vokabulars, der GQA-Konfiguration und der T4.

In [ ]:
# @title 2. Load and Quantize Model
#####################################
from transformers import AutoTokenizer, AutoModelForCausalLM

"""
Qwen2.5-7B-Instruct ist wie Mistral text-only, also bleibt die Grundstruktur identisch.
Wichtigster Unterschied zu Mistral v0.3: Qwens Chat-Template UNTERSTÜTZT die system-Rolle
(ChatML-Format: <|im_start|>system ... <|im_end|>\n<|im_start|>user ... <|im_end|>\n<|im_start|>assistant).
Du musst also, anders als bei Mistral, system-Content NICHT in den ersten User-Turn falten.

Zweiter Unterschied: Qwen2.5 hat ein ~5x größeres Vokabular (~152k statt Mistrals ~32k Tokens).
Das macht embed_tokens/lm_head anteilig größer und schiebt den LoRA-Trainable-Anteil leicht nach unten
(siehe Cell 4).
"""

# Select model (https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)
model_id = "Qwen/Qwen2.5-7B-Instruct"

# Qwen2.5-7B: ~15 GB auf Disk (bfloat16, ~7.61B Params inkl. größerem Vokabular)
# → ~5-5.5 GB nach 4-bit-Quantisierung (etwas mehr als Mistrals ~4 GB wegen der größeren Embedding-Matrix)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,   # ⚠️ geändert von bfloat16: T4 = Turing (sm_75),
                                              # hat KEINE BF16-Tensor-Core-Beschleunigung (die kam erst mit Ampere/sm_80+).
                                              # Auf einer A100/H100 wäre bfloat16 hier die bessere Wahl.
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer (text-only)
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Anders als Mistral bringt Qwen2.5 i.d.R. bereits einen definierten pad_token mit
    # (meist <|endoftext|>) — dieser Check greift hier also wahrscheinlich gar nicht, schadet aber nicht.

# Load and Quantize Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,        # ⚠️ ebenfalls float16 statt bfloat16, aus dem gleichen T4-Grund wie oben
    quantization_config=quant_config,
    device_map="auto")

In [ ]:
# @title 3. Setup LoRA Configuration (train only ~1% of parameters)
#####################################

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# target_modules bleiben EXAKT die gleichen Namen wie bei Mistral — Qwen2's Attention- und MLP-Klassen
# verwenden dieselbe Llama-Style-Benennung (q_proj/k_proj/v_proj/o_proj, gate_proj/up_proj/down_proj).
#
# Gotcha, das es bei Mistral nicht gibt: Qwen2 hat attention_bias=True für q_proj/k_proj/v_proj
# (o_proj bleibt bias-frei). Das ändert an dieser Config nichts — LoRA wrappt die Linear-Layer unabhängig
# davon, ob sie einen Bias-Term haben — aber du wirst in Cell 4 sehen, dass q/k/v_proj auch einen
# eingefrorenen `.bias`-Parameter mitschleppen, den Mistral so nicht hatte.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# @title 4. Analysis of Qwen Model
#####################################

print("\n ✅ Top-level submodules of the model:\n" + 40*"-")
for name, module in model.named_children():
    print(name, type(module).__name__)

print("\n ✅ Paths of inner submodules:\n" + 40*"-")
for name, module in model.model.named_children():
    print(name, type(module).__name__)

# Inspect the actual LoRA matrix shapes
# Erwartung für Qwen2.5-7B (hidden_size=3584, 28 Layer, GQA mit 4 KV-Heads bei 28 Q-Heads, head_dim=128):
#   q_proj: (3584, 3584) — quadratisch, wie bei Mistral, weil num_heads × head_dim = hidden_size
#   k_proj/v_proj: (512, 3584) statt quadratisch! num_key_value_heads(4) × head_dim(128) = 512
#     → das ist die GQA-Kompression: 28 Query-Heads teilen sich nur 4 Key/Value-Heads.
layer = model.base_model.model.model.layers[0].self_attn.q_proj
print("\n ✅ Inspect the actual LoRA matrix shapes:\n" + 40*"-")
print("base_layer:", layer.base_layer.weight.shape)      # erwartet: (3584, 3584)
print("lora_A:    ", layer.lora_A.default.weight.shape)  # erwartet: (16, 3584)
print("lora_B:    ", layer.lora_B.default.weight.shape)  # erwartet: (3584, 16)

# Quantify what LoRA actually costs
"""
Grobe Vorabschätzung (die exakte Zahl liefert dir der Print unten):
- 28 Layer × 7 target modules = 196 LoRA-injizierte Linear-Layer (Mistral: 224 bei 32 Layern)
- q_proj/o_proj: r×(in+out) = 16×(3584+3584) ≈ 115K je Layer
- k_proj/v_proj: wegen GQA kleiner: 16×(3584+512) ≈ 66K je Layer
- gate/up/down_proj (intermediate_size ≈ 18944): 16×(3584+18944) ≈ 360K je Layer, ×3
→ Summe ≈ 1.44M/Layer × 28 ≈ 40M trainierbare Params (Mistral hatte ~42M bei ähnlichem r)

Der %-Wert wird trotz ähnlicher absoluter Zahl NIEDRIGER liegen als Mistrals 1.1%:
Qwens ~5x größeres Vokabular (~152k vs. ~32k) macht embed_tokens + lm_head anteilig viel größer
(bei Qwen2.5-7B sind embed_tokens und lm_head NICHT tied, anders als bei den kleineren Qwen-Varianten),
das drückt den prozentualen LoRA-Anteil am Gesamtmodell nach unten — auch wenn absolut kaum weniger
Params trainiert werden.
"""
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("\n ✅ Quantify what LoRA actually costs:\n" + 40*"-")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

# Confirm quantization actually happened
print("\n ✅ Confirm quantization actually happened:\n" + 40*"-")
print(type(layer.base_layer).__name__)   # Linear4bit
print(layer.base_layer.weight.dtype)     # torch.uint8 (packed 4-bit)

# Print all modules
"""
Wie bei Mistral: Qwen2.5-7B-Instruct ist die reine Text-Variante (das Pendant zur Vision-Version wäre
Qwen2.5-VL, nicht dieses Modell). Also auch hier kein vision_tower/audio_tower — target_modules greift
ungequalifiziert überall dort, wo q_proj tatsächlich eine Attention-Projektion ist.

Neuer Gotcha ggü. Mistral: Der Layer-Count ist 28 statt 32 — falls du Code hast, der Layer-Indizes
hardcoded (z.B. layers[31] für "letzter Layer"), muss der auf layers[27] angepasst werden.
"""
print("\n" + 60*"=" + "\n ✅ Print all named modules to see the exact structure:\n" + 60*"=")
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)

# Check which params are actually frozen
"""
Anders als bei Mistral solltest du hier bei q_proj/k_proj/v_proj (nicht o_proj) auch ein `.bias`
sehen — aber NICHT unter den trainierbaren Params, da bias="none" in der LoraConfig sich nur auf
LoRAs eigene Biases bezieht und der Base-Layer-Bias durch prepare_model_for_kbit_training bereits
eingefroren wurde. Trainierbar sollten weiterhin nur lora_A/lora_B sein.
"""
print("\n" + 60*"=" + "\n ✅ Check which params are frozen:\n" + 60*"=")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, tuple(p.shape))

# Look at the chat template itself
"""
Hier der größte inhaltliche Unterschied zu Mistral: Qwen nutzt ChatML statt [INST]...[/INST], und
UNTERSTÜTZT die system-Rolle nativ:
<|im_start|>system\n...<|im_end|>\n<|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n...<|im_end|>
Falls dein Dataset also system-Turns enthält, brauchst du (anders als bei Mistral) KEINE
Umbau-Logik dafür.
"""
print("\n" + 60*"=" + "\n ✅ Print chat template:\n" + 60*"=")
print(tokenizer.chat_template)
print(tokenizer.apply_chat_template(
    [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}],
    tokenize=False,
))

Die genauen Zahlen (hidden_size, Layer-Anzahl, Vokabulargröße) hab ich aus dem, was ich über Qwen2.5-7B im Kopf habe – die solltest du dir einfach über die gedruckten Shapes bestätigen lassen, statt meinen Schätzungen blind zu vertrauen. Falls du statt Qwen2.5-7B-Instruct doch Qwen3-8B gewählt hast: nur model_id ändert sich, der Rest des Workflows bleibt praktisch identisch.